# Exercise Set 1

In [1]:
# Load libraries
import pandas as pd
import gurobipy as gb
from gurobipy import GRB

In [2]:
# Script for previous problem
production = ['Baltimore', 'Cleveland', 'Little_Rock', 'Birmingham', 'Charleston']
distribution = ['Columbia', 'Indianapolis', 'Lexington', 'Nashville', 'Richmond', 'St_Louis']
path = ""
transp_cost = pd.read_csv(path + "cost.csv", index_col=[0, 1]).squeeze()
max_prod = pd.Series([180, 200, 140, 80, 180], index=production, name="max_production")
n_demand = pd.Series([89, 95, 121, 101, 116, 181], index=distribution, name="demand")
frac = 0.75

m1 = gb.Model('widgets')

x = {}
for p in production:
    for d in distribution:
        x[p, d] = m1.addVar(name = p+"_to_"+d)
meet_demand = m1.addConstrs((gb.quicksum(x[p, d] for p in production) >= n_demand[d] for d in distribution),
                           name = "meet_demand")
can_produce = m1.addConstrs((gb.quicksum(x[p, d] for d in distribution) <= max_prod[p] for p in production),
                           name = "can_produce")
must_produce = m1.addConstrs((gb.quicksum(x[p, d] for d in distribution) >= frac * max_prod[p] for p in production),
                             name = "must_produce")
m1.setObjective(gb.quicksum(transp_cost[p, d] * x[p, d] for p in production for d in distribution),
               GRB.MINIMIZE)
m1.update()
m1.optimize()

Restricted license - for non-production use only - expires 2026-11-23
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 13th Gen Intel(R) Core(TM) i5-13450HX, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 16 rows, 30 columns and 90 nonzeros
Model fingerprint: 0x20186c14
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [2e+00, 7e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [6e+01, 2e+02]
Presolve removed 5 rows and 0 columns
Presolve time: 0.01s
Presolved: 11 rows, 35 columns, 65 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   1.610000e+02   0.000000e+00      0s
      15    1.7048900e+03   0.000000e+00   0.000000e+00      0s

Solved in 15 iterations and 0.02 seconds (0.00 work units)
Optimal objective  1.704890000e+03


You are told that there is a new policy for transporting **widgets** from production facilities. It is now required that the minimum number of widgets shipped from any production facility to any distribution center needs to be at least 20.

**Q1.** Write out how the formulation changes using mathematical notation given the new requirement.

$$
x_{p, d} \geq 20, \quad \forall \, p \in P, d \in D
$$

In [24]:
del m2
m2 = gb.Model('widgets')
min_shipment_amt = 20

x = {}
for p in production:
    for d in distribution:
        x[p, d] = m2.addVar(name = p+"_to_"+d)

meet_demand = m2.addConstrs((gb.quicksum(x[p, d] for p in production) >= n_demand[d] for d in distribution),
                           name = "meet_demand")
can_produce = m2.addConstrs((gb.quicksum(x[p, d] for d in distribution) <= max_prod[p] for p in production),
                           name = "can_produce")
must_produce = m2.addConstrs((gb.quicksum(x[p, d] for d in distribution) >= frac * max_prod[p] for p in production),
                            name = "must_produce")

must_ship = m2.addConstrs((x[p, d] >= min_shipment_amt for p in production for d in distribution), name = "must_ship")

m2.setObjective(gb.quicksum(transp_cost[p, d] * x[p, d] for p in production for d in distribution),
               GRB.MINIMIZE)
m2.update()
m2.optimize()

if m2.Status == GRB.OPTIMAL:
    # Get solution
    x_values = pd.Series(m2.getAttr('X', x), name="shipment", index=transp_cost.index)
    sol = pd.concat([transp_cost, x_values], axis=1)
    sol[sol.shipment > 0]
elif m2.Status == GRB.INF_OR_UNBD:
    print("Model is infeasible or unbounded")
elif m2.Status == GRB.INFEASIBLE:
    print("Model is infeasible")
elif m2.Status == GRB.UNBOUNDED:
    print("Model is unbounded")
else:
    print(f"Optimization ended with status {m2.Status}")

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 13th Gen Intel(R) Core(TM) i5-13450HX, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 46 rows, 30 columns and 120 nonzeros
Model fingerprint: 0xb8e1606f
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [2e+00, 7e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+01, 2e+02]
Presolve removed 34 rows and 4 columns
Presolve time: 0.01s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Infeasible model
Model is infeasible


The initial widget model $m1$ represented a single time period (e.g. a week, month, quarter). Suppose we added a time component using a set $T = \{0, 1, 2, 3\}$ representing a quarter of a year.

**Q2a.** Use `addVar()` or `addVars()` to create decision variables in gurobipy that represents the number of widgets shipped from a production facility to a distribution center for a given period of time.